# MADYS MCMC tutorial

This notebook demonstrates the MCMC sampling functions added in `madys.mcmc`.

We will:

1. Build a fast `IsochroneInterpolator` on top of a `madys.IsochroneGrid`.
2. Fit a single star's (age, mass) posterior with **emcee** using uniform-in-log priors bounded by the grid's dynamical range.
3. Run the same fit against two isochrone families -- **MIST** and **PARSEC2** -- and compare the posteriors.

> **Dependencies**: `emcee`, `corner`. Install with `pip install emcee corner` if not present.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from madys.madys import IsochroneGrid
from madys import mcmc

np.random.seed(42)

Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)

ImportError: cannot import name 'mcmc' from 'madys' (/ut3/jerome/miniconda3/envs/madys/lib/python3.8/site-packages/madys/__init__.py)

## 1. Pick a filter set and build an interpolator

We use Gaia DR3 (`G`, `Gbp`, `Grp`) and 2MASS (`J`, `H`, `K`), a combination supported by both MIST and PARSEC2. The interpolator fits a bivariate spline per filter across `(log_10 mass, log_10 age)`, so evaluating the model at arbitrary `(age, mass)` is cheap enough for the inner loop of an MCMC.

In [ ]:
filters = ['G', 'Gbp', 'Grp', 'J', 'H', 'K']

mass_range = (0.3, 2.0)    # Msun
age_range  = (5.0, 500.0)  # Myr
grid_steps = (150, 150)    # (n_mass, n_age) points in the interpolation grid

grid_mist    = IsochroneGrid('mist',    filters, mass_range=list(mass_range),
                             age_range=list(age_range), n_steps=list(grid_steps))
grid_parsec2 = IsochroneGrid('parsec2', filters, mass_range=list(mass_range),
                             age_range=list(age_range), n_steps=list(grid_steps))

interp_mist    = mcmc.IsochroneInterpolator(grid_mist)
interp_parsec2 = mcmc.IsochroneInterpolator(grid_parsec2)

print(interp_mist)
print(interp_parsec2)

## 2. Inject a synthetic star

To have a ground truth to compare against, we draw an idealized star from the MIST grid at a given (age, mass), then add Gaussian photometric noise. This is the same recipe you would apply to real data: hand `run_mcmc` the absolute magnitudes and per-band 1-sigma errors.

In [2]:
truth_age_Myr  = 30.0
truth_mass_Msun = 0.8

la_true = np.log10(truth_age_Myr)
lm_true = np.log10(truth_mass_Msun)

truth_mags = interp_mist.predict(la_true, lm_true)
phot_err   = np.array([0.02, 0.02, 0.02, 0.03, 0.03, 0.03])  # mag

rng = np.random.default_rng(42)
phot = truth_mags + phot_err * rng.standard_normal(truth_mags.shape)

print('filter:  ', '  '.join(f'{f:>4s}' for f in filters))
print('truth:   ', '  '.join(f'{m:>4.2f}' for m in truth_mags))
print('observed:', '  '.join(f'{m:>4.2f}' for m in phot))

NameError: name 'interp_mist' is not defined

## 3. Sample the posterior with MIST

`run_mcmc` returns the `emcee` sampler plus a post-burn, flattened chain ready for plotting. Reusing the pre-built `interpolator` skips the (slow) grid construction inside the call.

In [ ]:
res_mist = mcmc.run_mcmc(
    phot, phot_err, filters, model_version='mist',
    mass_range=mass_range, age_range=age_range,
    n_walkers=32, n_steps=3000, burn_in=1000,
    interpolator=interp_mist, seed=1, progress=False,
)

summary_mist = mcmc.mcmc_summary(res_mist)
for key, value in summary_mist.items():
    print(f'{key}: {value}')

### 3a. Check convergence with a trace plot

In [ ]:
def plot_traces(result, title=''):
    chain = result['sampler'].get_chain()  # shape: (n_steps, n_walkers, n_dim)
    fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
    labels = [r'$\log_{10}(\mathrm{age/Myr})$', r'$\log_{10}(M/M_\odot)$']
    for i, ax in enumerate(axes):
        ax.plot(chain[:, :, i], alpha=0.3, lw=0.5)
        ax.set_ylabel(labels[i])
    axes[-1].set_xlabel('step')
    fig.suptitle(title)
    fig.tight_layout()
    return fig

plot_traces(res_mist, title='MIST traces');

### 3b. Corner plot

In [ ]:
mcmc.plot_corner(
    res_mist, truths=[la_true, lm_true], show_titles=True, title_fmt='.3f',
    quantiles=[0.16, 0.5, 0.84],
);

## 4. Repeat with PARSEC2 and compare

`compare_models` is a convenience wrapper that calls `run_mcmc` once per model version and attaches a `summary` entry to each result.

In [ ]:
results = mcmc.compare_models(
    phot, phot_err, filters,
    model_versions=['mist', 'parsec2'],
    mass_range=mass_range, age_range=age_range,
    n_walkers=32, n_steps=3000, burn_in=1000, seed=1,
    n_grid_steps=grid_steps,
)

print(f'truth: age = {truth_age_Myr:.1f} Myr, mass = {truth_mass_Msun:.2f} Msun')
print('-' * 60)
for model, res in results.items():
    s = res['summary']
    la = s['log10_age_Myr']
    lm = s['log10_mass_Msun']
    print(f'{model:>8s}: log_age = {la["median"]:.3f} +{la["upper_1sigma"]:.3f}/-{la["lower_1sigma"]:.3f} '
          f'(age = {s["age_Myr_median"]:6.1f} Myr), '
          f'log_mass = {lm["median"]:.3f} +{lm["upper_1sigma"]:.3f}/-{lm["lower_1sigma"]:.3f} '
          f'(mass = {s["mass_Msun_median"]:.3f} Msun)')

### 4a. Overlay the MIST and PARSEC2 posteriors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for model, color in zip(['mist', 'parsec2'], ['tab:blue', 'tab:orange']):
    samples = results[model]['samples']
    axes[0].hist(samples[:, 0], bins=60, density=True, histtype='step',
                 color=color, label=model)
    axes[1].hist(samples[:, 1], bins=60, density=True, histtype='step',
                 color=color, label=model)

for ax, truth, label in zip(
    axes, [la_true, lm_true],
    [r'$\log_{10}(\mathrm{age/Myr})$', r'$\log_{10}(M/M_\odot)$']
):
    ax.axvline(truth, color='k', linestyle='--', lw=1, label='truth')
    ax.set_xlabel(label)
    ax.set_ylabel('posterior density')
    ax.legend()

fig.tight_layout()

### 4b. Stacked corner plot (MIST vs PARSEC2)

In [ ]:
import corner

labels = [r'$\log_{10}(\mathrm{age/Myr})$', r'$\log_{10}(M/M_\odot)$']

fig = corner.corner(
    results['mist']['samples'], color='tab:blue', labels=labels,
    truths=[la_true, lm_true], truth_color='k',
    plot_datapoints=False, plot_density=False, fill_contours=True,
    hist_kwargs={'density': True},
)
corner.corner(
    results['parsec2']['samples'], color='tab:orange', fig=fig,
    plot_datapoints=False, plot_density=False, fill_contours=True,
    hist_kwargs={'density': True},
)
fig.suptitle('MIST (blue) vs PARSEC2 (orange)', y=1.02);

### 4c. Compare the best-fit spectra

Use the posterior medians to predict the absolute magnitudes under each model and compare them to the observed data.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(filters))

ax.errorbar(x, phot, yerr=phot_err, fmt='ks', capsize=3, label='observed')

for model, color, interp in [
    ('mist', 'tab:blue', interp_mist),
    ('parsec2', 'tab:orange', interp_parsec2),
]:
    s = results[model]['summary']
    pred = interp.predict(s['log10_age_Myr']['median'], s['log10_mass_Msun']['median'])
    ax.plot(x, pred, 'o--', color=color,
            label=f"{model}: {s['age_Myr_median']:.1f} Myr, {s['mass_Msun_median']:.2f} Msun")

ax.set_xticks(x)
ax.set_xticklabels(filters)
ax.set_ylabel('absolute magnitude')
ax.invert_yaxis()
ax.legend()
fig.tight_layout()

## 5. Notes and tips

- **Prior**: `run_mcmc` uses a uniform-in-log prior restricted to the intersection of the grid dynamical range and the `mass_range` / `age_range` you pass in. To use a different prior, build the `IsochroneInterpolator` yourself and call `emcee.EnsembleSampler` with your own `log_probability`.
- **Extra parameters**: each model's chemistry/rotation/spots are fixed by the kwargs forwarded to `IsochroneGrid` (`feh`, `v_vcrit`, `fspot`, `B`, `he`, `afe`). For a non-solar fit, pass e.g. `feh=-0.5` through `run_mcmc` / `compare_models`.
- **Using a real `SampleObject`**: once you have queried the catalogs and `SampleObject.abs_phot` is populated, `mcmc.mcmc_from_sample(sample_obj, index, 'mist')` runs the fit on row `index`, masking filters whose error exceeds `ph_cut` (default `0.2` mag).
- **Model disagreement**: in practice, the (small) offsets between MIST and PARSEC2 posteriors reflect physical differences in opacities, convection prescriptions, and model atmospheres. Comparing multiple families like this is a light-weight form of systematic-error budgeting.